# multi_strat_050

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (75).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 10 |
| Detected functions | range_detector, process_group, process_ticker, backtest_trades, range_detector, plot_range_boxes_with_trades, range_detector, process_group, process_ticker, backtest_trades, hypertune_parameters |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/5min_data.csv')
data

In [ ]:
data['Ticker'].unique()

In [ ]:
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2022-01-01'
end_date = '2024-12-30'

# Define the tickers you want to filter
selected_tickers = ['ADANIENT', 'WIPRO', 'TATASTEEL', 'CIPLA', 'SBIN',
       'INDUSINDBK', 'BHARTIARTL', 'ONGC', 'DRREDDY', 'TRENT']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data

In [ ]:
data.dropna()
data

In [ ]:
data.columns

In [ ]:
from tqdm import tqdm
from joblib import Parallel, delayed
import pandas as pd
import numpy as np

# --------------------------------------------
# RANGE DETECTOR (Group by Ticker)
# --------------------------------------------
def range_detector(data, length=5, mult=2, atr_len=500, volume_mult=1.5):
    import numpy as np
    import pandas as pd

    def process_group(group):
        if 'volume' not in group.columns:
            raise ValueError("Volume column is required for volume confirmation.")

        group = group.sort_values("Date")
        close = group['close'].to_numpy()
        high = group['high'].to_numpy()
        low = group['low'].to_numpy()
        vol = group['volume'].to_numpy()

        prev_close = np.roll(close, 1)
        prev_close[0] = close[0]

        tr = np.maximum.reduce([
            high - low,
            np.abs(high - prev_close),
            np.abs(low - prev_close)
        ])
        atr = pd.Series(tr).rolling(atr_len).mean().to_numpy()
        sma = pd.Series(close).rolling(length).mean().to_numpy()
        upper = sma + atr * mult
        lower = sma - atr * mult

        # Outlier detection
        close_mat = np.lib.stride_tricks.sliding_window_view(close, window_shape=length)
        sma_mat = np.lib.stride_tricks.sliding_window_view(sma, window_shape=length)
        atr_mat = np.lib.stride_tricks.sliding_window_view(atr, window_shape=length)

        outlier_mask = np.abs(close_mat - sma_mat) > atr_mat * mult
        outlier_counts = np.concatenate([np.full(length - 1, np.nan), outlier_mask.sum(axis=1)])
        in_range = (outlier_counts == 0)

        # Volume rolling average
        avg_vol = pd.Series(vol).rolling(length).mean().to_numpy()
        volume_ok = vol > avg_vol * volume_mult

        # Generate signals
        signal = np.zeros(len(group), dtype=np.int8)
        box_top = box_bottom = None
        in_box = False

        for i in range(len(group)):
            if in_range[i] and not in_box:
                in_box = True
                box_top = upper[i]
                box_bottom = lower[i]
            elif not in_range[i] and in_box:
                in_box = False
            elif in_box:
                box_top = max(box_top, upper[i])
                box_bottom = min(box_bottom, lower[i])

            if box_top is not None and box_bottom is not None and volume_ok[i]:
                if close[i] > box_top:
                    signal[i] = 1
                elif close[i] < box_bottom:
                    signal[i] = 2

        # Add results to DataFrame
        group['H-L'] = high - low
        group['H-PC'] = np.abs(high - prev_close)
        group['L-PC'] = np.abs(low - prev_close)
        group['TR'] = tr
        group['ATR'] = atr
        group['SMA'] = sma
        group['Upper'] = upper
        group['Lower'] = lower
        group['Outliers'] = outlier_counts
        group['InRange'] = in_range
        group['VolumeAvg'] = avg_vol
        group['VolumeOK'] = volume_ok
        group['signal'] = signal

        return group

    return data.groupby('Ticker', group_keys=False).apply(process_group)

# --------------------------------------------
# PROCESS SINGLE TICKER
# --------------------------------------------
def process_ticker(ticker, ticker_data, initial_capital, brokerage, slippage, tp_pct, sl_pct, end_time, stop_trading_time):
    open_positions = {}
    trades = []
    cumulative_pnl = 0

    for row in ticker_data:
        timestamp, close, open_price, high, low, signal = row[1], row[5], row[2], row[3], row[4], row[17]

        if ticker in open_positions:
            position = open_positions[ticker]
            entry_price, direction, qty = position['entry_price'], position['direction'], position['qty']
            exit_price, exit_reason = None, None

            tp_price = entry_price * (1 + tp_pct) if direction == 'long' else entry_price * (1 - tp_pct)
            sl_price = entry_price * (1 - sl_pct) if direction == 'long' else entry_price * (1 + sl_pct)

            if (direction == 'long' and high >= tp_price) or (direction == 'short' and low <= tp_price):
                exit_price, exit_reason = tp_price, 'Take Profit'
            elif (direction == 'long' and low <= sl_price) or (direction == 'short' and high >= sl_price):
                exit_price, exit_reason = sl_price, 'Stop Loss'
            elif str(timestamp.astype('datetime64[m]'))[-5:] >= end_time:
                exit_price, exit_reason = close, 'EOD Close'

            if exit_price is not None:
                exit_price *= (1 - slippage) if direction == 'long' else (1 + slippage)
                raw_pnl = (exit_price - entry_price) * qty if direction == 'long' else (entry_price - exit_price) * qty
                pnl = raw_pnl - (brokerage * initial_capital * 2)
                cumulative_pnl += pnl

                trades.append([
                    ticker, position['entry_time'], timestamp, entry_price, exit_price,
                    pnl, exit_reason, cumulative_pnl, direction
                ])
                del open_positions[ticker]
                continue

        if ticker not in open_positions and str(timestamp.astype('datetime64[m]'))[-5:] < stop_trading_time:
            if signal == 1:
                direction = 'long'
            elif signal == 2:
                direction = 'short'
            else:
                continue

            entry_price = open_price * (1 + slippage) if direction == 'long' else open_price * (1 - slippage)
            qty = initial_capital / entry_price
            open_positions[ticker] = {
                'entry_time': timestamp,
                'entry_price': entry_price,
                'direction': direction,
                'qty': qty
            }

    return trades

# --------------------------------------------
# BACKTEST FUNCTION
# --------------------------------------------
def backtest_trades(data):
    initial_capital = 100000
    brokerage = 0.001
    slippage = 0.0001
    tp_pct = 0.04
    sl_pct = 0.005
    end_time = '15:25'
    stop_trading_time = '15:15'

    unique_tickers = data['Ticker'].unique()

    dtype = [
        ('Ticker', 'U10'), ('Date', 'M8[ms]'), ('open', 'f8'), ('high', 'f8'),
        ('low', 'f8'), ('close', 'f8'), ('volume', 'f8'), ('H-L', 'f8'),
        ('H-PC', 'f8'), ('L-PC', 'f8'), ('TR', 'f8'), ('ATR', 'f8'),
        ('SMA', 'f8'), ('Upper', 'f8'), ('Lower', 'f8'), ('Outliers', 'f8'),
        ('InRange', 'b1'), ('signal', 'i4'), ('RangeTop', 'f8'), ('RangeBottom', 'f8')
    ]

    data_np = np.array(
        list(data[['Ticker', 'Date', 'open', 'high', 'low', 'close', 'volume', 'H-L', 'H-PC', 'L-PC',
                   'TR', 'ATR', 'SMA', 'Upper', 'Lower', 'Outliers', 'InRange', 'signal', 'Upper', 'Lower']]
             .itertuples(index=False, name=None)),
        dtype=dtype
    )

    results = Parallel(n_jobs=-1)(
        delayed(process_ticker)(
            ticker,
            data_np[data_np['Ticker'] == ticker],
            initial_capital, brokerage, slippage, tp_pct, sl_pct, end_time, stop_trading_time
        ) for ticker in tqdm(unique_tickers)
    )

    trades = [trade for result in results for trade in result]
    return pd.DataFrame(trades, columns=[
        'Ticker', 'Entry Time', 'Exit Time', 'Entry Price', 'Exit Price',
        'PnL', 'Exit Reason', 'Cumulative PnL', 'Direction'
    ])

# --------------------------------------------
# USAGE
# --------------------------------------------
# Load your data
# data = pd.read_csv("your_data.csv")
data['Date'] = pd.to_datetime(data['Date'])

# Detect ranges and generate signals
data = range_detector(data)

# Run backtest
tradebook = backtest_trades(data)

# Save to file
tradebook.to_csv("/content/drive/MyDrive/tradebook_breakout_strategy.csv", index=False)

# Show output
tradebook


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

def range_detector(data, length=5, mult=2, atr_len=500):
    data = data.copy()

    # Calculate ATR
    data['H-L'] = data['high'] - data['low']
    data['H-PC'] = np.abs(data['high'] - data['close'].shift(1))
    data['L-PC'] = np.abs(data['low'] - data['close'].shift(1))
    data['TR'] = data[['H-L', 'H-PC', 'L-PC']].max(axis=1)
    data['ATR'] = data['TR'].rolling(atr_len, min_periods=1).mean()

    # Calculate SMA
    data['SMA'] = data['close'].rolling(length, min_periods=1).mean()

    # Define upper/lower band
    data['Upper'] = data['SMA'] + data['ATR'] * mult
    data['Lower'] = data['SMA'] - data['ATR'] * mult

    # Count outliers
    outliers = [np.nan] * (length - 1)
    for i in range(length - 1, len(data)):
        window = data.iloc[i - length + 1:i + 1]
        count = np.sum(np.abs(window['close'] - window['SMA']) > window['ATR'] * mult)
        outliers.append(count)
    data['Outliers'] = outliers
    data['InRange'] = (data['Outliers'] == 0)

    # Track range state
    state = []
    curr_state = 0
    for i in range(len(data)):
        if not data['InRange'].iloc[i]:
            state.append(curr_state)
            continue
        if data['close'].iloc[i] > data['Upper'].iloc[i]:
            curr_state = 1
        elif data['close'].iloc[i] < data['Lower'].iloc[i]:
            curr_state = -1
        else:
            curr_state = 0
        state.append(curr_state)

    data['State'] = state
    return data

def plot_range_boxes_with_trades(data, tradebook):
    fig = go.Figure()

    # Plot close price
    fig.add_trace(go.Scatter(x=data['Date'], y=data['close'],
                             mode='lines', name='Close', line=dict(color='black')))

    # Colors for state boxes
    state_colors = {
        0: 'rgba(33, 87, 243, 0.2)',   # Unbroken
        1: 'rgba(8, 153, 129, 0.2)',   # Broken Up
        -1: 'rgba(242, 54, 69, 0.2)'   # Broken Down
    }

    box_start = None
    for i in range(1, len(data)):
        prev = data['InRange'].iloc[i - 1]
        curr = data['InRange'].iloc[i]

        if curr and not prev:
            box_start = data['Date'].iloc[i]
        elif not curr and prev and box_start is not None:
            box_end = data['Date'].iloc[i]
            range_data = data[(data['Date'] >= box_start) & (data['Date'] <= box_end)]
            y0 = range_data['Lower'].min()
            y1 = range_data['Upper'].max()
            dominant_state = range_data['State'].mode().iloc[0] if not range_data['State'].mode().empty else 0

            fig.add_shape(type="rect", x0=box_start, x1=box_end, y0=y0, y1=y1,
                          fillcolor=state_colors[dominant_state], opacity=0.3, line=dict(width=0))
            box_start = None

    # Final open range
    if box_start is not None:
        box_end = data['Date'].iloc[-1]
        range_data = data[data['Date'] >= box_start]
        y0 = range_data['Lower'].min()
        y1 = range_data['Upper'].max()
        dominant_state = range_data['State'].mode().iloc[0] if not range_data['State'].mode().empty else 0
        fig.add_shape(type="rect", x0=box_start, x1=box_end, y0=y0, y1=y1,
                      fillcolor=state_colors[dominant_state], opacity=0.3, line=dict(width=0))

    # Overlay range lines
    fig.add_trace(go.Scatter(x=data['Date'], y=data['Upper'],
                             mode='lines', name='Range Top', line=dict(color='gray', dash='dash')))
    fig.add_trace(go.Scatter(x=data['Date'], y=data['Lower'],
                             mode='lines', name='Range Bottom', line=dict(color='gray', dash='dash')))

    # Plot trades
    for _, row in tradebook.iterrows():
        entry_time = row['Entry Time']
        exit_time = row['Exit Time']
        entry_price = row['Entry Price']
        exit_price = row['Exit Price']
        direction = row['Direction'].lower()
        pnl = row['PnL']

        color = 'blue' if direction == 'long' else 'red'
        symbol = 'triangle-up' if direction == 'long' else 'triangle-down'

        fig.add_trace(go.Scatter(
            x=[entry_time], y=[entry_price],
            mode='markers',
            marker=dict(color=color, size=8, symbol=symbol),
            name=direction.capitalize(),
            showlegend=False
        ))

        fig.add_trace(go.Scatter(
            x=[entry_time, exit_time], y=[entry_price, exit_price],
            mode='lines', line=dict(color='green' if pnl > 0 else 'red', width=2),
            showlegend=False
        ))

    fig.update_layout(
        title='Range Detector with Trade Signals',
        xaxis_title='Date',
        yaxis_title='Price',
        template='plotly_white',
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        height=700
    )

    fig.show()

# === Example Usage ===
# Load your data with datetime index
# data = pd.read_csv('your_price_data.csv', parse_dates=['datetime'])
# data.set_index('datetime', inplace=True)

# Load tradebook
# tradebook = pd.read_csv('your_tradebook.csv', parse_dates=['Entry Time', 'Exit Time'])

# Run
data = range_detector(data)
plot_range_boxes_with_trades(data, tradebook)


In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed
from itertools import product
import uuid

def range_detector(data, length=5, mult=2, atr_len=500, volume_mult=1.5):

    def process_group(group):
        if 'volume' not in group.columns:
            raise ValueError("Volume column is required for volume confirmation.")

        group = group.sort_values("Date")
        close = group['close'].to_numpy()
        high = group['high'].to_numpy()
        low = group['low'].to_numpy()
        vol = group['volume'].to_numpy()

        prev_close = np.roll(close, 1)
        prev_close[0] = close[0]

        tr = np.maximum.reduce([
            high - low,
            np.abs(high - prev_close),
            np.abs(low - prev_close)
        ])
        atr = pd.Series(tr).rolling(atr_len).mean().to_numpy()
        sma = pd.Series(close).rolling(length).mean().to_numpy()
        upper = sma + atr * mult
        lower = sma - atr * mult

        # Outlier detection
        close_mat = np.lib.stride_tricks.sliding_window_view(close, window_shape=length)
        sma_mat = np.lib.stride_tricks.sliding_window_view(sma, window_shape=length)
        atr_mat = np.lib.stride_tricks.sliding_window_view(atr, window_shape=length)

        outlier_mask = np.abs(close_mat - sma_mat) > atr_mat * mult
        outlier_counts = np.concatenate([np.full(length - 1, np.nan), outlier_mask.sum(axis=1)])
        in_range = (outlier_counts == 0)

        # Volume rolling average
        avg_vol = pd.Series(vol).rolling(length).mean().to_numpy()
        volume_ok = vol > avg_vol * volume_mult

        # Generate signals
        signal = np.zeros(len(group), dtype=np.int8)
        box_top = box_bottom = None
        in_box = False

        for i in range(len(group)):
            if in_range[i] and not in_box:
                in_box = True
                box_top = upper[i]
                box_bottom = lower[i]
            elif not in_range[i] and in_box:
                in_box = False
            elif in_box:
                box_top = max(box_top, upper[i])
                box_bottom = min(box_bottom, lower[i])

            if box_top is not None and box_bottom is not None and volume_ok[i]:
                if close[i] > box_top:
                    signal[i] = 1
                elif close[i] < box_bottom:
                    signal[i] = 2

        # Add results to DataFrame
        group['H-L'] = high - low
        group['H-PC'] = np.abs(high - prev_close)
        group['L-PC'] = np.abs(low - prev_close)
        group['TR'] = tr
        group['ATR'] = atr
        group['SMA'] = sma
        group['Upper'] = upper
        group['Lower'] = lower
        group['Outliers'] = outlier_counts
        group['InRange'] = in_range
        group['VolumeAvg'] = avg_vol
        group['VolumeOK'] = volume_ok
        group['signal'] = signal

        return group

    return data.groupby('Ticker', group_keys=False).apply(process_group)

def process_ticker(ticker, ticker_data, initial_capital, brokerage, slippage, tp_pct, sl_pct, end_time, stop_trading_time):
    open_positions = {}
    trades = []
    cumulative_pnl = 0

    for row in ticker_data:
        timestamp, close, open_price, high, low, signal = row["date"], row["close"], row["open"], row["high"], row["low"], row["signal"]
        if ticker in open_positions:
            position = open_positions[ticker]
            entry_price, direction, qty = position["entry_price"], position["direction"], position["qty"]
            exit_price, exit_reason = None, None
            tp_price = entry_price * (1 + tp_pct) if direction == "long" else entry_price * (1 - tp_pct)
            sl_price = entry_price * (1 - sl_pct) if direction == "long" else entry_price * (1 + sl_pct)
            if (direction == "long" and high >= tp_price) or (direction == "short" and low <= tp_price):
                exit_price, exit_reason = tp_price, "Take Profit"
            elif (direction == "long" and low <= sl_price) or (direction == "short" and high >= sl_price):
                exit_price, exit_reason = sl_price, "Stop Loss"
            elif timestamp.astype('datetime64[m]').astype(str)[-5:] >= end_time:
                exit_price, exit_reason = close, "EOD Close"
            if exit_price is not None:
                exit_price *= (1 - slippage) if direction == "long" else (1 + slippage)
                raw_pnl = (exit_price - entry_price) * qty if direction == "long" else (entry_price - exit_price) * qty
                pnl = raw_pnl - (brokerage * initial_capital * 2)
                cumulative_pnl += pnl
                trades.append([ticker, position["entry_time"], timestamp, entry_price, exit_price, pnl, exit_reason, cumulative_pnl, direction])
                del open_positions[ticker]
                continue
        if ticker not in open_positions and timestamp.astype('datetime64[m]').astype(str)[-5:] < stop_trading_time:
            if signal == 1:
                direction = "long"
            elif signal == 2:
                direction = "short"
            else:
                continue
            entry_price = open_price * (1 + slippage) if direction == "long" else open_price * (1 - slippage)
            qty = initial_capital / entry_price
            open_positions[ticker] = {"entry_time": timestamp, "entry_price": entry_price, "direction": direction, "qty": qty}
    return trades

def backtest_trades(data, params_length, atr_mult, volume_mult, tp_pct, sl_pct, atr_len):
    initial_capital = 100000
    brokerage = 0.001
    slippage = 0.0001
    end_time = "15:25"
    stop_trading_time = "15:15"

    # Calculate signals
    data_with_signals = range_detector(data, length=params_length, mult=atr_mult, atr_len=atr_len, volume_mult=volume_mult)

    dtype = [
        ("Ticker", "U10"), ("date", "M8[ms]"), ("close", "f8"),
        ("open", "f8"), ("high", "f8"), ("low", "f8"), ("signal", "i4")
    ]

    data_np = np.array(
        list(data_with_signals[["Ticker", "Date", "close", "open", "high", "low", "signal"]].itertuples(index=False, name=None)),
        dtype=dtype
    )

    unique_tickers = data['Ticker'].unique()
    results = Parallel(n_jobs=-1)(
        delayed(process_ticker)(
            ticker, data_np[data_np['Ticker'] == ticker], initial_capital, brokerage, slippage, tp_pct, sl_pct, end_time, stop_trading_time
        ) for ticker in unique_tickers
    )

    trades = [trade for result in results for trade in result]
    tradebook = pd.DataFrame(trades, columns=["Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price", "PnL", "Exit Reason", "Cumulative PnL", "Direction"])

    if tradebook.empty:
        return tradebook, 0, 0

    cumulative_pnl = tradebook["Cumulative PnL"].iloc[-1]
    cumulative_pnl_series = tradebook["Cumulative PnL"]
    peaks = cumulative_pnl_series.cummax()
    drawdowns = peaks - cumulative_pnl_series
    max_drawdown = drawdowns.max() if drawdowns.max() > 0 else 1
    profit_to_drawdown = cumulative_pnl / max_drawdown if max_drawdown != 0 else 0
    return tradebook, cumulative_pnl, profit_to_drawdown

def hypertune_parameters(data):
    data['Date'] = pd.to_datetime(data['Date'])

    start_date = '2022-01-01'
    end_date = '2024-12-31'

    selected_tickers = [
        'ADANIENT', 'WIPRO', 'TATASTEEL', 'CIPLA', 'SBIN',
       'INDUSINDBK', 'BHARTIARTL', 'ONGC', 'DRREDDY', 'TRENT'
    ]

    data = data[
        (data["Date"] >= start_date) &
        (data["Date"] <= end_date) &
        (data["Ticker"].isin(selected_tickers))
    ]

    param_grid = {
        'params_length': [5],
        'atr_mult': [1.75],
        'volume_mult': [1.5],
        'tp_pct': [0.04, 0.05],
        'sl_pct': [0.005, 0.015],
        'atr_len': [450, 500]
    }

    keys, values = zip(*param_grid.items())
    combinations = list(product(*values))
    total_combinations = len(combinations)

    best_profit = -float('inf')
    best_profit_params = None
    best_profit_drawdown = -float('inf')
    best_profit_drawdown_params = None

    print(f"Testing {total_combinations} parameter combinations...")

    for i, params in enumerate(tqdm(combinations, desc="Hypertuning Progress")):
        param_dict = dict(zip(keys, params))
        params_length, atr_mult, volume_mult, tp_pct, sl_pct, atr_len = params

        tradebook, cumulative_pnl, profit_to_drawdown = backtest_trades(
            data, params_length, atr_mult, volume_mult, tp_pct, sl_pct, atr_len
        )

        print(f"\nCombination {i+1}/{total_combinations}:")
        print(f"Parameters: {param_dict}")
        print(f"Cumulative PnL: {cumulative_pnl:.2f}")
        print(f"Profit-to-Drawdown Ratio: {profit_to_drawdown:.2f}")

        if cumulative_pnl > best_profit:
            best_profit = cumulative_pnl
            best_profit_params = param_dict

        if profit_to_drawdown > best_profit_drawdown:
            best_profit_drawdown = profit_to_drawdown
            best_profit_drawdown_params = param_dict

    print("\nBest Parameters for Maximum Profit:")
    print(f"Parameters: {best_profit_params}")
    print(f"Cumulative PnL: {best_profit:.2f}")

    print("\nBest Parameters for Profit-to-Drawdown Ratio:")
    print(f"Parameters: {best_profit_drawdown_params}")
    print(f"Profit-to-Drawdown Ratio: {best_profit_drawdown:.2f}")

if __name__ == "__main__":
    data = pd.read_csv("/content/drive/MyDrive/5min_data.csv")
    hypertune_parameters(data)
